In [1]:
"""
Compare OCR-extracted lab data against the hand-entered master dataset.

For every rabbit ID that appears in BOTH workbooks, the data table (row 14
headers, values from row 15 down) is compared cell by cell. The script then
reports:

  * rabbit IDs in the master that are missing from the comparison workbook
  * accuracy, precision, recall and F1 of the comparison workbook

Usage:
    Easiest: type your file names into MASTER_FILE / COMPARISON_FILE below,
    then run the script (press Run in your editor, or `python compare_lab_extraction.py`).

    Or pass them on the command line instead:
    python compare_lab_extraction.py master.xlsx comparison.xlsx --report results.xlsx


How each cell is scored
-----------------------
The master is treated as the truth.

    Master value | Comparison value | Result
    -------------+------------------+------------------------------------------
    value        | same value       | True Positive   (correct extraction)
    value        | different value  | Wrong value     (counts as 1 FP and 1 FN)
    value        | blank            | False Negative  (value was missed)
    blank        | value            | False Positive  (value was made up)
    blank        | blank            | True Negative   (correctly left empty)

    Accuracy  = cells that agree / all cells compared
    Precision = TP / (TP + FP)   "When it extracted a value, how often was it right?"
    Recall    = TP / (TP + FN)   "Of the real values, how many did it get right?"
    F1        = harmonic mean of precision and recall
"""

import argparse
import re
from dataclasses import dataclass, field
from datetime import datetime, time

import openpyxl


# ---------------------------------------------------------------------------
# File names: type your files here
# ---------------------------------------------------------------------------
# Use the full path if the files are not in the same folder as this script,
# e.g. r"C:\Users\Nik\Documents\CMPLT_Comp_Logs.xlsx"  (keep the r in front).
# Anything given on the command line overrides these.

MASTER_FILE = "CMPLT_Comp_Logs_ScenarioB.xlsx"
COMPARISON_FILE = "Processed_PDFs_11-06_-16-55.xlsx"
REPORT_FILE = "comparison_report_V3.xlsx"    # set to None to skip the Excel report


# ---------------------------------------------------------------------------
# Settings: change these if the template layout changes
# ---------------------------------------------------------------------------

ID_CELL_ROW = 4           # "Rabbit ID" label is in A4, the ID itself is in B4
HEADER_ROW = 14           # Row holding the column names (Timepoint, Clock Time, MAP, ...)
FIRST_DATA_ROW = 15       # First row of actual measurements

# Columns that are labels rather than measured values, so they are not scored.
COLUMNS_TO_SKIP = {"Timepoint", "Measurement >","Clock Time"}


# Two numbers count as equal if they match to this many decimal places.
DECIMAL_PLACES = 6


# ---------------------------------------------------------------------------
# Reading the workbooks
# ---------------------------------------------------------------------------

def load_rabbit_sheets(path):
    """
    Return {rabbit_id: list_of_rows} for every sheet that is a rabbit data sheet.

    A sheet counts as a rabbit sheet if cell A4 says "Rabbit ID". This
    automatically skips debugging sheets such as "Review Queue" or
    "Raw Extraction", whatever they are called.
    """
    # read_only=True avoids a crash on workbooks with broken image/drawing links.
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    sheets = {}

    for sheet in workbook.worksheets:
        # Some generated files store a wrong sheet size, which makes the sheet
        # look empty. Resetting it forces openpyxl to read every row.
        sheet.reset_dimensions()
        rows = [list(row) for row in sheet.iter_rows(values_only=True)]

        id_label = get_cell(rows, ID_CELL_ROW, 1)
        if str(id_label).strip().lower() != "rabbit id":
            continue

        # Prefer the ID written in B4; fall back to the sheet name.
        rabbit_id = get_cell(rows, ID_CELL_ROW, 2) or sheet.title
        sheets[str(rabbit_id).strip()] = rows

    workbook.close()
    return sheets


def get_cell(rows, row_number, column_number):
    """Safely read a cell using 1-based Excel row/column numbers."""
    if row_number > len(rows):
        return None
    row = rows[row_number - 1]
    if column_number > len(row):
        return None
    return row[column_number - 1]


# ---------------------------------------------------------------------------
# Matching rabbit IDs between the two workbooks
# ---------------------------------------------------------------------------

def simplify_id(rabbit_id):
    """
    Reduce an ID to its letters and digits, in lower case, with word breaks
    kept as single spaces. Example: "AEM FO-5" and "AEM FO 5" both become
    "aem fo 5".
    """
    words = re.split(r"[^A-Za-z0-9.]+", rabbit_id)
    return " ".join(word.lower() for word in words if word)


def match_ids(master_ids, comparison_ids):

    unmatched_comparison = {simplify_id(cid): cid for cid in comparison_ids}
    matched_pairs = []
    missing = []

    # Pass 1: exact matches
    leftovers = []
    for master_id in master_ids:
        key = simplify_id(master_id)
        if key in unmatched_comparison:
            matched_pairs.append((master_id, unmatched_comparison.pop(key)))
        else:
            leftovers.append(master_id)

    # Pass 2: whole-word prefix matches
    for master_id in leftovers:
        master_key = simplify_id(master_id)
        partner = None
        for comparison_key, comparison_id in unmatched_comparison.items():
            if (master_key.startswith(comparison_key + " ")
                    or comparison_key.startswith(master_key + " ")):
                partner = comparison_key
                break

        if partner is None:
            missing.append(master_id)
        else:
            matched_pairs.append((master_id, unmatched_comparison.pop(partner)))

    extra = list(unmatched_comparison.values())
    return matched_pairs, missing, extra


# ---------------------------------------------------------------------------
# Making values comparable
# ---------------------------------------------------------------------------

def is_blank(value):
    return value is None or str(value).strip() == ""


def normalize_value(value):
    """Turn a cell into something that can be compared with ==."""
    if is_blank(value):
        return None

    if isinstance(value, (int, float)):
        return round(float(value), DECIMAL_PLACES)

    text = str(value).strip()
    try:
        return round(float(text), DECIMAL_PLACES)   # "12.8" -> 12.8
    except ValueError:
        return text.lower()


# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------

@dataclass
class Score:
    true_positive: int = 0    # value present and correct
    true_negative: int = 0    # both blank
    wrong_value: int = 0      # both have a value, but they differ
    missed: int = 0           # master has a value, comparison is blank
    made_up: int = 0          # master is blank, comparison has a value
    mismatches: list = field(default_factory=list)

    @property
    def false_positive(self):
        return self.made_up + self.wrong_value

    @property
    def false_negative(self):
        return self.missed + self.wrong_value

    @property
    def total_cells(self):
        return (self.true_positive + self.true_negative
                + self.wrong_value + self.missed + self.made_up)

    @property
    def accuracy(self):
        correct = self.true_positive + self.true_negative
        return safe_divide(correct, self.total_cells)

    @property
    def precision(self):
        return safe_divide(self.true_positive, self.true_positive + self.false_positive)

    @property
    def recall(self):
        return safe_divide(self.true_positive, self.true_positive + self.false_negative)

    @property
    def f1(self):
        p, r = self.precision, self.recall
        return safe_divide(2 * p * r, p + r)

    def add(self, other):
        """Add another Score's counts into this one."""
        self.true_positive += other.true_positive
        self.true_negative += other.true_negative
        self.wrong_value += other.wrong_value
        self.missed += other.missed
        self.made_up += other.made_up
        self.mismatches.extend(other.mismatches)


# Labels for the three kinds of error
WRONG = "Wrong value"
MISSED = "Missed"
MADE_UP = "Made up"

# Columns shown in the error lists (printed and in the Excel report)
ERROR_COLUMNS = ["Model", "Variable", "Excel row", "Master value", "Comparison value"]


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def last_row_with_data(rows):
    """Excel row number of the last row (at/after FIRST_DATA_ROW) containing anything."""
    for row_number in range(len(rows), FIRST_DATA_ROW - 1, -1):
        if any(not is_blank(cell) for cell in rows[row_number - 1]):
            return row_number
    return FIRST_DATA_ROW - 1


def compare_sheets(rabbit_id, master_rows, comparison_rows):
    """Compare one rabbit's data table cell by cell and return a Score."""
    score = Score()
    headers = master_rows[HEADER_ROW - 1]
    last_row = max(last_row_with_data(master_rows), last_row_with_data(comparison_rows))

    for column_number, header in enumerate(headers, start=1):
        if is_blank(header) or str(header).strip() in COLUMNS_TO_SKIP:
            continue

        header = str(header).strip()
        normalize = normalize_value 

        for row_number in range(FIRST_DATA_ROW, last_row + 1):
            master_raw = get_cell(master_rows, row_number, column_number)
            comparison_raw = get_cell(comparison_rows, row_number, column_number)
            expected = normalize(master_raw)
            extracted = normalize(comparison_raw)

            if expected is None and extracted is None:
                score.true_negative += 1
                continue
            if expected == extracted:
                score.true_positive += 1
                continue

            if expected is None:
                score.made_up += 1
                problem = MADE_UP
            elif extracted is None:
                score.missed += 1
                problem = MISSED
            else:
                score.wrong_value += 1
                problem = WRONG

            score.mismatches.append({
                "Model": rabbit_id,
                "Variable": header,
                "Excel row": row_number,
                "Master value": master_raw,
                "Comparison value": comparison_raw,
                "Problem": problem,
            })

    return score


# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------

def print_results(missing, extra, matched_pairs, per_rabbit, overall):
    print("=" * 60)
    print("MISSING FROM COMPARISON DATASET")
    print("=" * 60)
    if missing:
        for rabbit_id in missing:
            print(f"  - {rabbit_id}")
    else:
        print("  None. Every master ID was found.")

    if extra:
        print("\nIn comparison dataset but not in master (not scored):")
        for rabbit_id in extra:
            print(f"  - {rabbit_id}")

    renamed = [(m, c) for m, c in matched_pairs if m != c]
    if renamed:
        print("\nIDs matched despite different spelling:")
        for master_id, comparison_id in renamed:
            print(f"  - master '{master_id}'  <->  comparison '{comparison_id}'")

    print("\n" + "=" * 60)
    print("PER-RABBIT RESULTS")
    print("=" * 60)
    print(f"  {'Rabbit ID':<24}{'Accuracy':>10}{'Precision':>11}{'Recall':>9}{'F1':>9}")
    for rabbit_id, score in per_rabbit.items():
        print(f"  {rabbit_id:<24}{score.accuracy:>10.1%}{score.precision:>11.1%}"
              f"{score.recall:>9.1%}{score.f1:>9.1%}")

    print("\n" + "=" * 60)
    print(f"OVERALL ({len(per_rabbit)} rabbits, {overall.total_cells} cells compared)")
    print("=" * 60)
    print(f"  Accuracy : {overall.accuracy:.2%}")
    print(f"  Precision: {overall.precision:.2%}")
    print(f"  Recall   : {overall.recall:.2%}")
    print(f"  F1 score : {overall.f1:.2%}")
    print(f"\n  Correct values: {overall.true_positive}   Correct blanks: {overall.true_negative}")
    print(f"  Wrong values: {overall.wrong_value}   Missed: {overall.missed}   "
          f"Made up: {overall.made_up}")


def display(value):
    """Show a cell value as text; blanks are shown as '(blank)'."""
    return "(blank)" if is_blank(value) else str(value)


def print_error_lists(overall):
    """Print every wrong, missed and made-up value, grouped by type of error."""
    descriptions = {
        WRONG: "both files have a value, but they differ",
        MISSED: "master has a value, comparison is blank",
        MADE_UP: "master is blank, comparison has a value",
    }

    for problem, description in descriptions.items():
        errors = [m for m in overall.mismatches if m["Problem"] == problem]

        print("\n" + "=" * 60)
        print(f"{problem.upper()} ({len(errors)}): {description}")
        print("=" * 60)
        if not errors:
            print("  None.")
            continue

        print(f"  {'Model':<24}{'Variable':<12}{'Row':>5}   {'Master':<14}{'Comparison':<14}")
        for error in errors:
            print(f"  {error['Model']:<24}{error['Variable']:<12}{error['Excel row']:>5}   "
                  f"{display(error['Master value']):<14}"
                  f"{display(error['Comparison value']):<14}")


def add_error_sheet(workbook, sheet_name, errors):
    """Add one sheet listing the given errors, one per row."""
    sheet = workbook.create_sheet(sheet_name)
    sheet.append(ERROR_COLUMNS)
    for error in errors:
        row = []
        for column in ERROR_COLUMNS:
            value = error[column]
            # Keep numbers as numbers (so Excel can sort/filter them); mark blanks
            row.append("(blank)" if is_blank(value) else value)
        sheet.append(row)
    # Widen columns so the values are readable when the file is opened
    for letter, width in zip("ABCDE", [26, 14, 10, 16, 18]):
        sheet.column_dimensions[letter].width = width


def write_report(path, missing, extra, per_rabbit, overall):
    """Save the summary, per-rabbit scores and every mismatching cell to Excel."""
    workbook = openpyxl.Workbook()

    summary = workbook.active
    summary.title = "Summary"
    summary.append(["Metric", "Value"])
    for name in ["accuracy", "precision", "recall", "f1"]:
        summary.append([name.capitalize(), round(getattr(overall, name), 4)])
    summary.append(["Cells compared", overall.total_cells])
    summary.append([])
    summary.append(["Missing from comparison"] + missing)
    summary.append(["Not in master"] + extra)

    by_rabbit = workbook.create_sheet("Per Rabbit")
    by_rabbit.append(["Rabbit ID", "Accuracy", "Precision", "Recall", "F1",
                      "Correct values", "Correct blanks", "Wrong values",
                      "Missed", "Made up"])
    for rabbit_id, s in per_rabbit.items():
        by_rabbit.append([rabbit_id, round(s.accuracy, 4), round(s.precision, 4),
                          round(s.recall, 4), round(s.f1, 4), s.true_positive,
                          s.true_negative, s.wrong_value, s.missed, s.made_up])

    # One sheet per kind of error
    for problem, sheet_name in [(WRONG, "Wrong Values"), (MISSED, "Missed"),
                                (MADE_UP, "Made Up")]:
        errors = [m for m in overall.mismatches if m["Problem"] == problem]
        add_error_sheet(workbook, sheet_name, errors)

    workbook.save(path)
    print(f"\nDetailed report saved to: {path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(
        description="Compare OCR-extracted lab data against the master dataset.")
    parser.add_argument("master", nargs="?", default=MASTER_FILE,
                        help="Master workbook (default: MASTER_FILE at top of script)")
    parser.add_argument("comparison", nargs="?", default=COMPARISON_FILE,
                        help="Extracted workbook (default: COMPARISON_FILE at top of script)")
    parser.add_argument("--report", default=REPORT_FILE,
                        help="Excel path for a detailed report (default: REPORT_FILE)")
    # parse_known_args (instead of parse_args) ignores extra arguments that
    # Jupyter notebooks add behind the scenes, e.g. "--f=...kernel.json".
    args, _ = parser.parse_known_args()

    master_sheets = load_rabbit_sheets(args.master)
    comparison_sheets = load_rabbit_sheets(args.comparison)

    matched_pairs, missing, extra = match_ids(list(master_sheets), list(comparison_sheets))

    per_rabbit = {}
    overall = Score()
    for master_id, comparison_id in matched_pairs:
        score = compare_sheets(master_id, master_sheets[master_id],
                               comparison_sheets[comparison_id])
        per_rabbit[master_id] = score
        overall.add(score)

    print_results(missing, extra, matched_pairs, per_rabbit, overall)
    print_error_lists(overall)

    if args.report:
        write_report(args.report, missing, extra, per_rabbit, overall)


if __name__ == "__main__":
    main()

MISSING FROM COMPARISON DATASET
  None. Every master ID was found.

PER-RABBIT RESULTS
  Rabbit ID                 Accuracy  Precision   Recall       F1
  B-EA 10                      95.7%      98.7%    92.7%    95.6%
  B-EPS 01                     98.9%      98.8%    98.2%    98.5%
  B-EPS 02                     98.8%      98.5%    97.8%    98.2%
  B-EPS 03                     96.8%      96.6%    93.3%    94.9%
  B-EPS 04                     99.6%      99.4%    99.4%    99.4%
  B-EPS 05                     98.6%      99.4%    97.5%    98.4%
  B-EPS 08                     99.2%      98.8%    98.8%    98.8%
  B-EPS 09                     98.6%      98.1%    97.5%    97.8%
  B-EA 02                      98.4%      97.1%    97.1%    97.1%
  B-EA 07                      98.2%      97.5%    96.9%    97.2%
  B-EAS 01                     98.6%      98.8%    97.5%    98.1%
  B-EAS 02                     97.1%      96.3%    95.1%    95.7%
  B-EPS 06                     97.1%      96.1%    94.8